In [ ]:
import pandas as pd
import numpy as np
import math
import warnings
import plotly.express as px
# Chia thành 2 biểu đồ trong cùng một figure
from plotly.subplots import make_subplots
import seaborn as sns
warnings.filterwarnings('ignore')

DATA_DIR  = '../data/raw/'

## **Part 1. Descriptive — "What happened?**

Các bảng dữ liệu sử dụng: sales.csv, orders.csv, products.csv, customers.csv

**1. Doanh thu tổng quan**
- Revenue theo ngày/tháng/năm
- Gross profit margin
- COGS vs Revenue trend
- Biểu đồ sử dụng: Line chart + area chart

**2. Sản phẩm & khách hàng**
- Top sản phẩm / category
- Phân bố segment, size
- Số KH theo age_group
- Biểu đồ sử dụng: Bar chart + pie chart

**3. Vận hành**
- Tỷ lệ trả hàng / huỷ đơn
- Web traffic theo source
- Rating phân bố 1–5 sao
- Biểu đồ sử dụng: Histogram + heatmap

In [ ]:
# 1. Doanh thu tổng quan
# Revenue theo ngày tháng năm
sales_df = pd.read_csv(DATA_DIR + 'sales.csv')
sales_df['Date'] = pd.to_datetime(sales_df['Date'])
total_revenue = sales_df['Revenue'].sum()

# vẽ biểu đồ doanh thu theo thời gian bằng plotly
daily_revenue = sales_df.groupby('Date')['Revenue'].sum().reset_index()
fig = px.line(daily_revenue, x='Date', y='Revenue', title='Doanh thu theo ngày')
fig.update_layout(xaxis_title='Ngày', yaxis_title='Doanh thu (VND)')
fig.show()

# Gross profit margin theo ngày tháng năm (lấy theo median để giảm ảnh hưởng của outliers)
sales_df['Gross Profit Margin'] = (sales_df['Revenue'] - sales_df['COGS']) / sales_df['Revenue']
daily_gpm = sales_df.groupby('Date')['Gross Profit Margin'].median().reset_index()
fig = px.line(daily_gpm, x='Date', y='Gross Profit Margin', title='Gross Profit Margin theo ngày')
fig.update_layout(xaxis_title='Ngày', yaxis_title='Gross Profit Margin')
fig.show()

# COGS and Revenue trend (area chart)
daily_cogs_revenue = sales_df.groupby('Date')[['COGS', 'Revenue']].sum().reset_index()

# Tạo biểu đồ vùng với chế độ không cộng dồn
fig = px.area(daily_cogs_revenue, 
              x='Date', 
              y=['COGS', 'Revenue'], 
              title='Phân tích Lợi nhuận gộp: Doanh thu vs Giá vốn',
              color_discrete_map={'Revenue': 'blue', 'COGS': 'red'})

# Chỉnh độ trong suốt (opacity) để nhìn thấy phần giao nhau
fig.update_traces(opacity=0.5)
fig.update_layout(barmode='overlay', hovermode='x unified')
fig.show()

#

In [31]:
# 2. Sản phẩm & khách hàng**
# Load dữ liệu sản phẩm và khách hàng
products_df = pd.read_csv(DATA_DIR + 'products.csv')
customers_df = pd.read_csv(DATA_DIR + 'customers.csv')
orders_item_df = pd.read_csv(DATA_DIR + 'order_items.csv')
orders_df = pd.read_csv(DATA_DIR + 'orders.csv')
# Vẽ Top 5 sản phẩm được bán nhiều nhất theo category
# Tính tổng revenue trong orders_items sau đó merge với products để lấy category
orders_item_df['Revenue'] = orders_item_df['quantity'] * orders_item_df['unit_price']
product_revenue = orders_item_df.groupby('product_id')['Revenue'].sum().reset_index()
product_revenue = product_revenue.merge(products_df[['product_id', 'category']], on='product_id', how='left')
category_revenue = product_revenue.groupby('category')['Revenue'].sum().reset_index()
fig = px.bar(category_revenue, x='category', y='Revenue', title='Doanh thu theo category sản phẩm')
fig.update_layout(xaxis_title='Category', yaxis_title='Doanh thu (VND)')
fig.show()

# Phân bố segment theo đơn hàng/ số lượng sản phẩm
# Biểu đồ sử dụng: Pie chart
# merge cột segment từ products vào orders_item để đếm số lượng đơn hàng theo segment
orders_item_df = orders_item_df.merge(products_df[['product_id', 'segment']], on='product_id', how='left')
segment_counts_by_order = orders_item_df['segment'].value_counts().reset_index()
segment_counts_by_order.columns= ['segment', 'count']

# Phân bố segment theo quantity sản phẩm
segment_quantity = orders_item_df.groupby('segment')['quantity'].sum().reset_index()

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]], subplot_titles=['Số lượng đơn hàng theo segment', 'Số lượng sản phẩm theo segment'])
fig.add_trace(px.pie(segment_counts_by_order, values='count', names='segment').data[0], row=1, col=1)
fig.add_trace(px.pie(segment_quantity, values='quantity', names='segment').data[0], row=1, col=2)
fig.update_traces(textposition='inside', textinfo='percent+label') 
fig.update_layout(title_text='Phân khúc thị trường theo segment', showlegend=False)
fig.show()

# Phân bố size
# merge cột size từ products vào orders_item để đếm số lượng đơn hàng theo size
orders_item_df = orders_item_df.merge(products_df[['product_id', 'size']], on='product_id', how='left')
size_counts_by_order = orders_item_df['size'].value_counts().reset_index() 
size_counts_by_order.columns= ['size', 'count']
# Đếm số lượng sản phẩm theo size
size_quantity = orders_item_df.groupby('size')['quantity'].sum().reset_index()

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'domain'}, {'type':'domain'}]], subplot_titles=['Số lượng đơn hàng theo size', 'Số lượng sản phẩm theo size'])
fig.add_trace(px.pie(size_counts_by_order, values='count', names='size').data[0], row=1, col=1)
fig.add_trace(px.pie(size_quantity, values='quantity', names='size').data[0], row=1, col=2)
fig.update_traces(textposition='inside', textinfo='percent+label') 
fig.update_layout(title_text='Phân bố size sản phẩm', showlegend=False)
fig.show()

#Số KH theo age_group (bar chart)
age_group_counts = customers_df['age_group'].value_counts().reset_index()
age_group_counts.columns = ['age_group', 'count']
fig = px.bar(age_group_counts, x='age_group', y='count', title='Số lượng khách hàng theo nhóm tuổi')
fig.update_layout(xaxis_title='Nhóm tuổi', yaxis_title='Số lượng khách hàng')
fig.show()

# Tổng số đơn hàng theo nhóm tuổi (bar chart) kèm theo đường line chỉ số đơn hàng trung bình theo nhóm tuổi (line chart)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Chuẩn bị dữ liệu
# Nối bảng để lấy thông tin nhóm tuổi
df_merged = orders_df.merge(customers_df[['customer_id', 'age_group']], on='customer_id', how='left')
df_merged = df_merged[df_merged['age_group'].notnull()] # Loại bỏ giá trị null theo yêu cầu Q6 

# Tính tổng đơn hàng và số khách hàng duy nhất mỗi nhóm
age_stats = df_merged.groupby('age_group').agg(
    total_orders=('order_id', 'nunique'),
    unique_customers=('customer_id', 'nunique')
).reset_index()

# Tính trung bình (Số đơn / Số khách) - Đây là chỉ số quan trọng cho Q6 [cite: 170]
age_stats['avg_orders'] = age_stats['total_orders'] / age_stats['unique_customers']
age_stats = age_stats.sort_values('age_group') # Đảm bảo thứ tự nhóm tuổi đúng

# 2. Vẽ biểu đồ Dual Axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Thêm Bar Chart (Tổng đơn hàng)
fig.add_trace(
    go.Bar(x=age_stats['age_group'], y=age_stats['total_orders'], name="Tổng số đơn hàng"),
    secondary_y=False,
)

# Thêm Line Chart (Trung bình mỗi khách)
fig.add_trace(
    go.Scatter(x=age_stats['age_group'], y=age_stats['avg_orders'], name="Đơn hàng TB/Khách", mode='lines+markers'),
    secondary_y=True,
)

fig.update_layout(title_text='Phân tích hành vi mua sắm theo Nhóm tuổi')
fig.update_xaxes(title_text='Nhóm tuổi')
fig.update_yaxes(title_text='Tổng số đơn hàng (Quy mô)', secondary_y=False)
fig.update_yaxes(title_text='Số đơn/Khách (Lòng trung thành)', secondary_y=True)
fig.show()

# Doanh thu theo nhóm tuổi (bar chart) kèm theo đường line chỉ số doanh thu trung bình theo nhóm tuổi (line chart)
# Tính doanh thu theo nhóm tuổi cần merge thêm cột Revenue có sẵn từ orders_item_df vào df_merged
df_merged = df_merged.merge(orders_item_df[['order_id', 'Revenue']], on='order_id', how='left')
age_revenue_stats = df_merged.groupby('age_group').agg(
    total_revenue=('Revenue', 'sum'),
    avg_revenue=('Revenue', 'mean')
).reset_index()
age_revenue_stats = age_revenue_stats.sort_values('age_group') # Đảm bảo thứ tự nhóm tuổi đúng
fig = make_subplots(specs=[[{"secondary_y": True}]])
# Thêm Bar Chart (Tổng doanh thu)
fig.add_trace(
    go.Bar(x=age_revenue_stats['age_group'], y=age_revenue_stats['total_revenue'], name="Tổng doanh thu"),
    secondary_y=False,
)
# Thêm Line Chart (Doanh thu trung bình mỗi khách)
fig.add_trace(
    go.Scatter(x=age_revenue_stats['age_group'], y=age_revenue_stats['avg_revenue'], name="Doanh thu TB/Khách", mode='lines+markers'),
    secondary_y=True,
)  
fig.update_layout(title_text='Phân tích doanh thu theo Nhóm tuổi')
fig.update_xaxes(title_text='Nhóm tuổi')
fig.update_yaxes(title_text='Tổng doanh thu (Quy mô)', secondary_y=False)
fig.update_yaxes(title_text='Doanh thu/Khách (Lòng trung thành)', secondary_y=True)
fig.show()


Bà Khanh dới Kim Ngăn coi tới đây thui nhen. Khúc dưới tui thấy data noise rùi

In [41]:
# Xem có case nào return_quantity > quantity gốc không
check = (
    returns_df
    .groupby(['order_id', 'product_id'])['return_quantity'].sum()
    .reset_index()
    .merge(orders_item_df[['order_id', 'product_id', 'quantity']], 
           on=['order_id', 'product_id'])
)
check[check['return_quantity'] > check['quantity']]  # phải rỗng

,order_id,product_id,return_quantity,quantity
19649,397622,2045,11,8
19650,397622,2045,11,6
33781,698128,1009,4,1


In [ ]:
# 3. Vận hành
returns_df = pd.read_csv(DATA_DIR + 'returns.csv')
# Tỷ lệ trả lại (tính tổng return_id trong returns.csv so với tổng số đơn hàng trong orders.csv)
return_rate = returns_df['return_id'].nunique() / orders_df['order_id'].nunique()
# Tỷ lệ trả lại > 2 lần (đếm số order_id xuất hiện > 2 lần trong returns.csv)
returned_order_counts = returns_df['order_id'].value_counts()
returned_order_counts = returned_order_counts[returned_order_counts > 2]
# Vẽ biểu đồ phân bố số lần trả lại của các đơn hàng (histogram) kèm đường line return_rate theo từng order_id (line chart)
fig = px.histogram(returns_df, x='order_id', title='Phân bố số lần trả lại của các đơn hàng')
fig.update_layout(xaxis_title='Order ID', yaxis_title='Số lần trả lại')
fig.show()

# Tỷ lệ return
# Web traffic theo source
# Rating phân bố 1–5 sao
# Biểu đồ sử dụng: Histogram + heatmap

In [32]:
orders_df['order_status'] = orders_df['order_status'].str.lower() # Chuẩn hóa chữ thường
# đếm returned 
returned_count = orders_df[orders_df['order_status'] == 'returned'].shape[0]
print(f"Số đơn hàng bị trả lại: {returned_count}")
total_orders = orders_df.shape[0]
return_rate = returned_count / total_orders if total_orders > 0 else 0

Số đơn hàng bị trả lại: 36142


In [37]:
returns_df = pd.read_csv(DATA_DIR + 'returns.csv')
returns_df.shape[0]
# đếm unique order_id trong returns để biết số đơn bị trả lại
unique_returned_orders = returns_df['order_id'].nunique()
print(f"Số đơn hàng bị trả lại: {unique_returned_orders}")
# in ra tất cả order_id bị trả lại > 2 lần và kèm theo số lần bị trả lại
returned_order_counts = returns_df['order_id'].value_counts()
returned_order_counts[returned_order_counts > 2]
print(returned_order_counts[returned_order_counts > 2])

Số đơn hàng bị trả lại: 36062
order_id
252224    3
191080    3
65261     3
319108    3
27504     3
         ..
667670    3
684619    3
689967    3
702501    3
795558    3
Name: count, Length: 76, dtype: int64
